# 08 Appendix Additional Ablations

This notebook collects the non-winning branches that still matter scientifically. They should stay out of the main text, but they are important for showing that the final thesis claims are not based on a shallow search.


In [ ]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

def _find_thesis_root():
    cwd = Path.cwd().resolve()
    direct = [cwd, *cwd.parents]
    nested = [candidate / "thesis" for candidate in direct]
    for candidate in [*direct, *nested]:
        if (
            (candidate / "src" / "qc_thesis" / "__init__.py").exists()
            and (candidate / "README.md").exists()
            and (candidate / "notebooks").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not find thesis root from notebook session")

ROOT = _find_thesis_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from qc_thesis import *

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)
apply_thesis_style()


def show_saved_figure(path, caption=None):
    figure_path = Path(path)
    if not figure_path.is_absolute():
        figure_path = (ROOT / figure_path).resolve()
    if not figure_path.exists():
        display(Markdown(f"_Missing figure: `{figure_path}`_"))
        return
    if caption:
        display(Markdown(caption))
    display(Image(filename=str(figure_path)))

fig_dir, table_dir = notebook_output_dirs("08_appendix_additional_ablations")
appendix_files = (
    build_recipe_inventory()
    .loc[lambda df: df["main_text"].fillna(False)]
    .loc[:, ["runner_key", "implementation_module", "recipe_id", "result_key"]]
    .drop_duplicates()
    .sort_values(["runner_key", "recipe_id"])
    .reset_index(drop=True)
)
fpos_groups = get_recipe_groups("fpos")
fmiss_groups = get_recipe_groups("fmiss")


## 1. Research module inventory

This is the live recipe-to-module inventory for the thesis modeling package.


In [ ]:
display(appendix_files)
save_table(appendix_files, table_dir, "research_module_inventory")


## 2. Ablation grouping reference

The appendix should still be organized by scientific question, not by raw script name. These group maps are the compact reference for the appendix chapter structure.


In [ ]:
appendix_fpos = pd.DataFrame([(k, ', '.join(v)) for k, v in fpos_groups.items()], columns=["ablation_group", "recipe_ids"])
appendix_fmiss = pd.DataFrame([(k, ', '.join(v)) for k, v in fmiss_groups.items()], columns=["ablation_group", "recipe_ids"])
display(appendix_fpos)
display(appendix_fmiss)
save_table(appendix_fpos, table_dir, "appendix_fpos_recipe_groups")
save_table(appendix_fmiss, table_dir, "appendix_fmiss_recipe_groups")


## 3. What belongs in appendix rather than main text

The scientific rule is simple:

- main text keeps the strongest benchmark ladder, target asymmetry, and robustness story
- appendix keeps negative branches, tuning branches, transport/synthetic branches, and exploratory dead ends that still matter for scientific completeness


In [ ]:
display(Markdown(
    """
**Appendix families to emphasize**

- synthetic augmentation and paired-conditioned transport
- residual correction and manifold-context variants
- waveform enrichment and multiscale waveform variants
- Bayesian tuning and meta/bagging branches
- tree interaction and paper-inspired transfer branches
"""
))
